# MiniExcel — an ipywidgets Excel alternative

A spreadsheet that runs inside a Jupyter notebook. Features:
- Editable grid with row/column headers (A, B, C / 1, 2, 3)
- Formulas: `=SUM`, `=AVERAGE`, `=MIN`, `=MAX`, `=COUNT`, `=PRODUCT`, `=ABS`, `=ROUND`, `=IF`, `=CONCAT`
- Arithmetic: `+ - * / ^`, parentheses, comparisons, negation
- Cell references (`A1`) and ranges (`A1:B5`)
- Automatic recalculation of dependent cells
- Undo / Redo
- Insert / delete rows and columns
- Load & save CSV / XLSX
- Sample data already loaded so you can play immediately

### Setup
Run this once if you don't already have the dependencies:
```
pip install ipywidgets pandas openpyxl
```
Then run the cells below in order.

In [8]:
# Cell 1: The spreadsheet engine.
# This is the same code as spreadsheet_engine.py but embedded for a self-contained notebook.

import re, csv, copy
from typing import Any

def col_letter(col):
    s, n = '', col
    while True:
        s = chr(ord('A') + n % 26) + s
        n = n // 26 - 1
        if n < 0: break
    return s

def col_index(letters):
    n = 0
    for ch in letters.upper():
        n = n * 26 + (ord(ch) - ord('A') + 1)
    return n - 1

def a1_to_rc(ref):
    m = re.fullmatch(r'([A-Za-z]+)(\d+)', ref.strip())
    if not m: raise ValueError(f'Bad cell reference: {ref}')
    letters, digits = m.groups()
    return int(digits) - 1, col_index(letters)

def rc_to_a1(row, col):
    return f'{col_letter(col)}{row + 1}'

class FormulaError(Exception): pass

def _product(xs):
    p = 1
    for x in xs: p *= x
    return p

FUNCTIONS = {
    'SUM': lambda xs: sum(xs),
    'AVERAGE': lambda xs: (sum(xs)/len(xs)) if xs else FormulaError('#DIV/0!'),
    'MIN': lambda xs: min(xs) if xs else FormulaError('#VALUE!'),
    'MAX': lambda xs: max(xs) if xs else FormulaError('#VALUE!'),
    'COUNT': lambda xs: len(xs),
    'PRODUCT': lambda xs: _product(xs),
    'ABS': lambda xs: abs(xs[0]) if len(xs)==1 else FormulaError('#VALUE!'),
    'ROUND': lambda xs: round(xs[0], int(xs[1])) if len(xs)==2 else FormulaError('#VALUE!'),
}

class Evaluator:
    def __init__(self, get_cell):
        self.get_cell = get_cell
        self.deps = set()
    def evaluate(self, text):
        self.deps = set()
        self.tokens = self._tokenize(text); self.pos = 0
        r = self._expr()
        if self.pos != len(self.tokens): raise FormulaError('#SYNTAX')
        return r
    def _tokenize(self, s):
        tokens, i = [], 0
        while i < len(s):
            c = s[i]
            if c.isspace(): i += 1
            elif c.isdigit() or (c=='.' and i+1<len(s) and s[i+1].isdigit()):
                j = i
                while j < len(s) and (s[j].isdigit() or s[j]=='.'): j += 1
                tokens.append(('NUM', float(s[i:j]))); i = j
            elif c == '"':
                j = i + 1
                while j < len(s) and s[j] != '"': j += 1
                tokens.append(('STR', s[i+1:j])); i = j + 1
            elif c.isalpha():
                j = i
                while j < len(s) and (s[j].isalnum() or s[j]=='_'): j += 1
                w = s[i:j]
                if re.fullmatch(r'[A-Za-z]+\d+', w): tokens.append(('REF', w.upper()))
                elif w.upper() in ('TRUE','FALSE'): tokens.append(('BOOL', w.upper()=='TRUE'))
                else: tokens.append(('FUNC', w.upper()))
                i = j
            elif c in '+-*/^(),:<>=':
                if c in '<>' and i+1<len(s) and s[i+1]=='=':
                    tokens.append(('OP', c+'=')); i += 2
                elif c == '<' and i+1<len(s) and s[i+1]=='>':
                    tokens.append(('OP', '<>')); i += 2
                else: tokens.append(('OP', c)); i += 1
            else: raise FormulaError(f"#SYNTAX: unexpected '{c}'")
        return tokens
    def _peek(self, o=0):
        return self.tokens[self.pos+o] if self.pos+o<len(self.tokens) else (None, None)
    def _eat(self):
        t = self.tokens[self.pos]; self.pos += 1; return t
    def _match(self, kind, val=None):
        t = self._peek()
        if t[0] == kind and (val is None or t[1] == val): return self._eat()
        return None
    def _expr(self):
        left = self._additive()
        while self._peek()[0]=='OP' and self._peek()[1] in ('=','<>','<','>','<=','>='):
            op = self._eat()[1]; right = self._additive()
            left = self._compare(left, op, right)
        return left
    def _compare(self, a, op, b):
        return {'=':a==b, '<>':a!=b, '<':a<b, '>':a>b, '<=':a<=b, '>=':a>=b}[op]
    def _additive(self):
        left = self._term()
        while self._peek()[0]=='OP' and self._peek()[1] in ('+','-'):
            op = self._eat()[1]; right = self._term()
            left = self._num(left)+self._num(right) if op=='+' else self._num(left)-self._num(right)
        return left
    def _term(self):
        left = self._power()
        while self._peek()[0]=='OP' and self._peek()[1] in ('*','/'):
            op = self._eat()[1]; right = self._power()
            if op == '*': left = self._num(left) * self._num(right)
            else:
                r = self._num(right)
                if r == 0: raise FormulaError('#DIV/0!')
                left = self._num(left) / r
        return left
    def _power(self):
        left = self._factor()
        if self._peek()[0]=='OP' and self._peek()[1]=='^':
            self._eat(); right = self._factor()
            return self._num(left) ** self._num(right)
        return left
    def _factor(self):
        t = self._peek()
        if t[0]=='OP' and t[1]=='-': self._eat(); return -self._num(self._factor())
        if t[0]=='OP' and t[1]=='+': self._eat(); return self._num(self._factor())
        if t[0]=='NUM': return self._eat()[1]
        if t[0]=='STR': return self._eat()[1]
        if t[0]=='BOOL': return self._eat()[1]
        if t[0]=='OP' and t[1]=='(':
            self._eat(); v = self._expr()
            if not self._match('OP', ')'): raise FormulaError('#SYNTAX: missing )')
            return v
        if t[0]=='REF':
            r1 = self._eat()[1]
            if self._peek()[0]=='OP' and self._peek()[1]==':':
                self._eat()
                if self._peek()[0] != 'REF': raise FormulaError('#SYNTAX')
                r2 = self._eat()[1]
                return self._resolve_range(r1, r2)
            return self._resolve_cell(r1)
        if t[0]=='FUNC':
            name = self._eat()[1]
            if not self._match('OP', '('): raise FormulaError(f'#NAME?: {name}')
            args = []
            if not (self._peek()[0]=='OP' and self._peek()[1]==')'):
                args.append(self._expr())
                while self._peek()[0]=='OP' and self._peek()[1]==',':
                    self._eat(); args.append(self._expr())
            if not self._match('OP', ')'): raise FormulaError(f'#SYNTAX: missing ) in {name}')
            return self._call(name, args)
        raise FormulaError('#SYNTAX: unexpected token')
    def _resolve_cell(self, ref):
        r, c = a1_to_rc(ref); self.deps.add((r, c))
        v = self.get_cell(r, c)
        if v is None or v == '': return 0
        if isinstance(v, str):
            try: return float(v)
            except ValueError: return v
        return v
    def _resolve_range(self, r1, r2):
        rr1, cc1 = a1_to_rc(r1); rr2, cc2 = a1_to_rc(r2)
        rs, re_ = min(rr1,rr2), max(rr1,rr2)
        cs, ce = min(cc1,cc2), max(cc1,cc2)
        vals = []
        for r in range(rs, re_+1):
            for c in range(cs, ce+1):
                self.deps.add((r, c))
                v = self.get_cell(r, c)
                if v is None or v == '': continue
                if isinstance(v, (int, float)): vals.append(float(v))
                elif isinstance(v, str):
                    try: vals.append(float(v))
                    except ValueError: pass
        return vals
    def _call(self, name, args):
        if name == 'IF':
            if len(args) != 3: raise FormulaError('#VALUE!')
            return args[1] if args[0] else args[2]
        if name in ('CONCAT','CONCATENATE'):
            return ''.join(str(a) for a in args)
        if name not in FUNCTIONS: raise FormulaError(f'#NAME?: {name}')
        flat = []
        for a in args:
            if isinstance(a, list): flat.extend(a)
            elif isinstance(a, (int, float)): flat.append(float(a))
            elif isinstance(a, str):
                try: flat.append(float(a))
                except ValueError: pass
            elif isinstance(a, bool): flat.append(1.0 if a else 0.0)
        r = FUNCTIONS[name](flat)
        if isinstance(r, FormulaError): raise r
        return r
    def _num(self, v):
        if isinstance(v, bool): return 1 if v else 0
        if isinstance(v, (int, float)): return v
        if isinstance(v, str):
            try: return float(v)
            except ValueError: raise FormulaError('#VALUE!')
        if isinstance(v, list): raise FormulaError('#VALUE!')
        raise FormulaError('#VALUE!')

class Spreadsheet:
    def __init__(self, rows=20, cols=10):
        self.rows, self.cols = rows, cols
        self.raw, self.values = {}, {}
        self.deps_of, self.dependents = {}, {}
        self._history, self._future = [], []
        self._history_limit = 50
    def _snapshot(self):
        return {'raw': dict(self.raw), 'values': dict(self.values),
                'deps_of': {k: set(v) for k,v in self.deps_of.items()},
                'dependents': {k: set(v) for k,v in self.dependents.items()},
                'rows': self.rows, 'cols': self.cols}
    def _restore(self, s):
        self.raw = dict(s['raw']); self.values = dict(s['values'])
        self.deps_of = {k: set(v) for k,v in s['deps_of'].items()}
        self.dependents = {k: set(v) for k,v in s['dependents'].items()}
        self.rows, self.cols = s['rows'], s['cols']
    def _push(self):
        self._history.append(self._snapshot())
        if len(self._history) > self._history_limit: self._history.pop(0)
        self._future.clear()
    def undo(self):
        if not self._history: return False
        self._future.append(self._snapshot()); self._restore(self._history.pop()); return True
    def redo(self):
        if not self._future: return False
        self._history.append(self._snapshot()); self._restore(self._future.pop()); return True
    def set_cell(self, row, col, text, record_history=True):
        if record_history: self._push()
        key = (row, col)
        if key in self.deps_of:
            for d in self.deps_of[key]: self.dependents.get(d, set()).discard(key)
            del self.deps_of[key]
        if text is None or text == '':
            self.raw.pop(key, None); self.values.pop(key, None)
        else:
            self.raw[key] = text
        self._recalc(row, col)
        self._recalc_deps(key)
    def _recalc(self, row, col):
        key = (row, col); text = self.raw.get(key, '')
        if text == '': self.values.pop(key, None); return
        if isinstance(text, str) and text.startswith('='):
            ev = Evaluator(lambda r, c: self.values.get((r, c)))
            try:
                r = ev.evaluate(text[1:])
                if isinstance(r, list): r = r[0] if r else ''
                self.values[key] = r
                self.deps_of[key] = ev.deps
                for d in ev.deps: self.dependents.setdefault(d, set()).add(key)
            except FormulaError as e: self.values[key] = str(e)
            except Exception: self.values[key] = '#ERROR!'
        else:
            try: self.values[key] = float(text) if '.' in text or 'e' in text.lower() else int(text)
            except (ValueError, TypeError): self.values[key] = text
    def _recalc_deps(self, key):
        seen, queue = set(), list(self.dependents.get(key, set()))
        while queue:
            c = queue.pop(0)
            if c in seen: continue
            seen.add(c); self._recalc(*c)
            queue.extend(self.dependents.get(c, set()))
    def recalculate_all(self):
        for _ in range(5):
            changed = False
            for k in list(self.raw.keys()):
                before = self.values.get(k); self._recalc(*k)
                if self.values.get(k) != before: changed = True
            if not changed: break
    def get_raw(self, r, c): return self.raw.get((r, c), '')
    def get_value(self, r, c): return self.values.get((r, c), '')
    def get_display(self, r, c):
        v = self.values.get((r, c), '')
        if isinstance(v, float):
            if v.is_integer(): return str(int(v))
            return f'{v:g}'
        return str(v)
    def insert_row(self, at):
        self._push()
        self.raw = {((r+1 if r>=at else r), c): v for (r,c), v in self.raw.items()}
        self.rows += 1; self._rebuild()
    def delete_row(self, at):
        self._push()
        self.raw = {((r-1 if r>at else r), c): v for (r,c), v in self.raw.items() if r != at}
        self.rows = max(1, self.rows-1); self._rebuild()
    def insert_col(self, at):
        self._push()
        self.raw = {(r, (c+1 if c>=at else c)): v for (r,c), v in self.raw.items()}
        self.cols += 1; self._rebuild()
    def delete_col(self, at):
        self._push()
        self.raw = {(r, (c-1 if c>at else c)): v for (r,c), v in self.raw.items() if c != at}
        self.cols = max(1, self.cols-1); self._rebuild()
    def _rebuild(self):
        self.values.clear(); self.deps_of.clear(); self.dependents.clear()
        for k in list(self.raw.keys()): self._recalc(*k)
        self.recalculate_all()
    def load_from_2d(self, data):
        self.raw.clear(); self.values.clear()
        self.deps_of.clear(); self.dependents.clear()
        if data:
            self.rows = max(self.rows, len(data))
            self.cols = max(self.cols, max(len(r) for r in data))
        for r, row in enumerate(data):
            for c, val in enumerate(row):
                if val is None or val == '': continue
                self.set_cell(r, c, str(val), record_history=False)
        self.recalculate_all()
        self._history.clear(); self._future.clear()
    def save_csv(self, path):
        data = [['' for _ in range(self.cols)] for _ in range(self.rows)]
        for (r,c), v in self.values.items():
            if r < self.rows and c < self.cols: data[r][c] = v
        with open(path, 'w', newline='') as f: csv.writer(f).writerows(data)
    def load_csv(self, path):
        with open(path, newline='') as f: rows = list(csv.reader(f))
        self.load_from_2d(rows)
    def save_xlsx(self, path):
        from openpyxl import Workbook
        wb = Workbook(); ws = wb.active
        for (r,c), text in self.raw.items():
            ws.cell(row=r+1, column=c+1, value=text)
        wb.save(path)
    def load_xlsx(self, path):
        from openpyxl import load_workbook
        wb = load_workbook(path); ws = wb.active
        data = [list(row) for row in ws.iter_rows(values_only=True)]
        self.load_from_2d(data)

print('Engine loaded.')

Engine loaded.


In [ ]:
# Cell 2: The ipywidgets UI
import ipywidgets as widgets
import csv, io
from IPython.display import display

class MiniExcelUI:
    def __init__(self, rows=12, cols=8):
        self.sheet = Spreadsheet(rows, cols)
        self.cell_widgets = {}
        self.col_header_widgets = {}
        self.selected = (0, 0)
        self._programmatic_update = False
        self.col_dropdowns = {}
        self.col_widths = {}
        self.hidden_cols = set()
        self.editable_cols = set()   # whitelist — empty = all columns locked
        self.page = 0
        self.page_size = 50
        self.freeze_top = False
        self.grid_height = 400
        self._build()

    def _col_w(self, c):
        return f'{self.col_widths.get(c, 90)}px'

    def _cell_layout(self, c):
        w = self._col_w(c)
        return widgets.Layout(width=w, min_width=w, flex='0 0 auto')

    def _apply_col_width(self, c, px):
        w_str = f'{px}px'
        if c in self.col_header_widgets:
            h = self.col_header_widgets[c]
            h.layout.width = w_str
            h.layout.min_width = w_str
        for r in range(self.sheet.rows):
            if (r, c) in self.cell_widgets:
                w = self.cell_widgets[(r, c)]
                w.layout.width = w_str
                w.layout.min_width = w_str

    def _apply_col_visibility(self, c, hidden):
        val = 'none' if hidden else ''
        if c in self.col_header_widgets:
            self.col_header_widgets[c].layout.display = val
        for r in range(self.sheet.rows):
            if (r, c) in self.cell_widgets:
                self.cell_widgets[(r, c)].layout.display = val

    def _build(self):
        # ── Row 0: Input data ──────────────────────────────────────────────
        self.input_area = widgets.Textarea(
            placeholder='Paste CSV or tab-separated data here, then click Load…',
            layout=widgets.Layout(width='560px', height='60px'))
        self.load_btn = widgets.Button(
            description='Load', button_style='info',
            layout=widgets.Layout(width='70px'))
        self.load_btn.on_click(self._on_load_data)
        toolbar0 = widgets.HBox([
            widgets.Label('Input data:', layout=widgets.Layout(width='80px')),
            self.input_area, self.load_btn,
        ])

        # ── Row 1: Save as ─────────────────────────────────────────────────
        self.path_input = widgets.Text(
            value='miniexcel_output', placeholder='filename (no extension)',
            layout=widgets.Layout(width='220px'))
        self.save_csv_btn  = widgets.Button(description='Save CSV',
                                            layout=widgets.Layout(width='90px'))
        self.save_xlsx_btn = widgets.Button(description='Save XLSX',
                                            layout=widgets.Layout(width='90px'))
        self.save_csv_btn.on_click(self._on_save_csv)
        self.save_xlsx_btn.on_click(self._on_save_xlsx)
        toolbar1 = widgets.HBox([
            widgets.Label('Save as:', layout=widgets.Layout(width='60px')),
            self.path_input, self.save_csv_btn, self.save_xlsx_btn,
        ])

        # ── Row 2: Formula bar ─────────────────────────────────────────────
        self.cell_label  = widgets.Label(value='A1',
                                         layout=widgets.Layout(width='50px'))
        self.formula_bar = widgets.Text(
            value='', continuous_update=False,
            placeholder='Enter value or formula (e.g. =SUM(A1:A5))',
            layout=widgets.Layout(width='500px'))
        self.formula_bar.observe(self._on_formula_bar_change, names='value')
        toolbar2 = widgets.HBox([self.cell_label, self.formula_bar])

        # ── Row 3: Edit buttons ────────────────────────────────────────────
        self.undo_btn    = widgets.Button(description='Undo', icon='undo',
                                          layout=widgets.Layout(width='90px'))
        self.redo_btn    = widgets.Button(description='Redo', icon='redo',
                                          layout=widgets.Layout(width='90px'))
        self.ins_row_btn = widgets.Button(description='+Row',
                                          tooltip='Insert row above selected',
                                          layout=widgets.Layout(width='70px'))
        self.del_row_btn = widgets.Button(description='-Row',
                                          tooltip='Delete selected row',
                                          layout=widgets.Layout(width='70px'))
        self.ins_col_btn = widgets.Button(description='+Col',
                                          tooltip='Insert column before selected',
                                          layout=widgets.Layout(width='70px'))
        self.del_col_btn = widgets.Button(description='-Col',
                                          tooltip='Delete selected column',
                                          layout=widgets.Layout(width='70px'))
        self.clear_btn   = widgets.Button(description='Clear All',
                                          button_style='warning',
                                          layout=widgets.Layout(width='100px'))
        self.undo_btn.on_click(self._on_undo)
        self.redo_btn.on_click(self._on_redo)
        self.ins_row_btn.on_click(self._on_ins_row)
        self.del_row_btn.on_click(self._on_del_row)
        self.ins_col_btn.on_click(self._on_ins_col)
        self.del_col_btn.on_click(self._on_del_col)
        self.clear_btn.on_click(self._on_clear)
        toolbar3 = widgets.HBox([
            self.undo_btn, self.redo_btn,
            self.ins_row_btn, self.del_row_btn,
            self.ins_col_btn, self.del_col_btn,
            self.clear_btn,
        ])

        # ── Row 4: Dropdown config ─────────────────────────────────────────
        self.add_dd_btn = widgets.Button(
            description='Add Dropdown', button_style='', icon='list',
            layout=widgets.Layout(width='130px'))
        self.add_dd_btn.on_click(self._on_toggle_dd_config)

        self.dd_col_input   = widgets.Text(placeholder='Column (e.g. A)',
                                           layout=widgets.Layout(width='100px'))
        self.dd_items_input = widgets.Text(placeholder='Items: Yes, No, Maybe',
                                           layout=widgets.Layout(width='260px'))
        self.dd_apply_btn   = widgets.Button(description='Apply', button_style='success',
                                             layout=widgets.Layout(width='70px'))
        self.dd_remove_btn  = widgets.Button(description='Remove', button_style='danger',
                                             layout=widgets.Layout(width='80px'))
        self.dd_apply_btn.on_click(self._on_apply_dropdown)
        self.dd_remove_btn.on_click(self._on_remove_dropdown)

        self.dd_config_row = widgets.HBox([
            widgets.Label('Column:', layout=widgets.Layout(width='58px')),
            self.dd_col_input,
            widgets.Label('Items:', layout=widgets.Layout(width='43px')),
            self.dd_items_input,
            self.dd_apply_btn, self.dd_remove_btn,
        ], layout=widgets.Layout(display='none'))

        toolbar4 = widgets.HBox([self.add_dd_btn, self.dd_config_row])

        # ── Row 5: Column width config ─────────────────────────────────────
        self.col_w_btn = widgets.Button(
            description='Column Width', button_style='', icon='arrows-h',
            layout=widgets.Layout(width='130px'))
        self.col_w_btn.on_click(self._on_toggle_cw_config)

        self.cw_col_input   = widgets.Text(placeholder='Column (e.g. A)',
                                           layout=widgets.Layout(width='100px'))
        self.cw_width_input = widgets.BoundedIntText(
            value=90, min=30, max=600, step=10,
            layout=widgets.Layout(width='75px'))
        self.cw_apply_btn   = widgets.Button(description='Apply', button_style='success',
                                             layout=widgets.Layout(width='70px'))
        self.cw_reset_btn   = widgets.Button(description='Reset',
                                             layout=widgets.Layout(width='70px'))
        self.cw_apply_btn.on_click(self._on_apply_col_width)
        self.cw_reset_btn.on_click(self._on_reset_col_width)

        self.cw_config_row = widgets.HBox([
            widgets.Label('Column:', layout=widgets.Layout(width='58px')),
            self.cw_col_input,
            widgets.Label('Width px:', layout=widgets.Layout(width='65px')),
            self.cw_width_input,
            self.cw_apply_btn, self.cw_reset_btn,
        ], layout=widgets.Layout(display='none'))

        toolbar5 = widgets.HBox([self.col_w_btn, self.cw_config_row])

        # ── Row 6: Hide / Unhide column ────────────────────────────────────
        self.hide_col_btn = widgets.Button(
            description='Hide/Unhide Col', button_style='', icon='eye-slash',
            layout=widgets.Layout(width='150px'))
        self.hide_col_btn.on_click(self._on_toggle_hide_config)

        self.hc_col_input  = widgets.Text(placeholder='Column (e.g. A)',
                                           layout=widgets.Layout(width='100px'))
        self.hc_hide_btn   = widgets.Button(description='Hide', button_style='warning',
                                             layout=widgets.Layout(width='70px'))
        self.hc_unhide_btn = widgets.Button(description='Unhide', button_style='success',
                                             layout=widgets.Layout(width='80px'))
        self.hc_hide_btn.on_click(self._on_hide_col)
        self.hc_unhide_btn.on_click(self._on_unhide_col)

        self.hc_config_row = widgets.HBox([
            widgets.Label('Column:', layout=widgets.Layout(width='58px')),
            self.hc_col_input,
            self.hc_hide_btn, self.hc_unhide_btn,
        ], layout=widgets.Layout(display='none'))

        toolbar6 = widgets.HBox([self.hide_col_btn, self.hc_config_row])

        # ── Row 7: Pagination ──────────────────────────────────────────────
        self.page_size_dd = widgets.Dropdown(
            options=[10, 20, 30, 40, 50],
            value=self.page_size,
            layout=widgets.Layout(width='65px'))
        self.page_size_dd.observe(self._on_page_size_change, names='value')

        self.prev_btn  = widgets.Button(description='◀ Prev',
                                        layout=widgets.Layout(width='85px'))
        self.next_btn  = widgets.Button(description='Next ▶',
                                        layout=widgets.Layout(width='85px'))
        self.page_label = widgets.Label(value='Page 1 of 1',
                                        layout=widgets.Layout(width='110px'))
        self.prev_btn.on_click(self._on_prev_page)
        self.next_btn.on_click(self._on_next_page)

        toolbar7 = widgets.HBox([
            widgets.Label('Rows/page:', layout=widgets.Layout(width='75px')),
            self.page_size_dd,
            self.prev_btn,
            self.page_label,
            self.next_btn,
        ])

        # ── Row 8: Freeze top row ──────────────────────────────────────────
        self.freeze_btn = widgets.ToggleButton(
            value=False, description='Freeze Top Row', icon='lock',
            layout=widgets.Layout(width='145px'))
        self.freeze_btn.observe(self._on_freeze_toggle, names='value')

        self.freeze_height_input = widgets.BoundedIntText(
            value=self.grid_height, min=100, max=800, step=50,
            layout=widgets.Layout(width='75px'))
        self.freeze_height_apply = widgets.Button(
            description='Apply', button_style='success',
            layout=widgets.Layout(width='70px'))
        self.freeze_height_apply.on_click(self._on_apply_grid_height)

        self.freeze_config_row = widgets.HBox([
            widgets.Label('Height px:', layout=widgets.Layout(width='72px')),
            self.freeze_height_input,
            self.freeze_height_apply,
        ], layout=widgets.Layout(display='none'))

        toolbar8 = widgets.HBox([self.freeze_btn, self.freeze_config_row])

        # ── Row 9: Column edit whitelist ───────────────────────────────────
        self.col_edit_btn = widgets.Button(
            description='Column Edit', button_style='', icon='pencil',
            layout=widgets.Layout(width='130px'))
        self.col_edit_btn.on_click(self._on_toggle_col_edit_config)

        self.ce_col_input      = widgets.Text(placeholder='Column (e.g. A)',
                                              layout=widgets.Layout(width='100px'))
        self.ce_enable_btn     = widgets.Button(description='Enable',
                                                button_style='success',
                                                layout=widgets.Layout(width='75px'))
        self.ce_disable_btn    = widgets.Button(description='Disable',
                                                button_style='warning',
                                                layout=widgets.Layout(width='75px'))
        self.ce_enable_all_btn = widgets.Button(description='Enable All',
                                                button_style='info',
                                                layout=widgets.Layout(width='90px'))
        self.ce_disable_all_btn= widgets.Button(description='Disable All',
                                                button_style='danger',
                                                layout=widgets.Layout(width='95px'))
        self.ce_enable_btn.on_click(self._on_enable_col_edit)
        self.ce_disable_btn.on_click(self._on_disable_col_edit)
        self.ce_enable_all_btn.on_click(self._on_enable_all_cols)
        self.ce_disable_all_btn.on_click(self._on_disable_all_cols)

        self.ce_config_row = widgets.HBox([
            widgets.Label('Column:', layout=widgets.Layout(width='58px')),
            self.ce_col_input,
            self.ce_enable_btn, self.ce_disable_btn,
            self.ce_enable_all_btn, self.ce_disable_all_btn,
        ], layout=widgets.Layout(display='none'))

        toolbar9 = widgets.HBox([self.col_edit_btn, self.ce_config_row])

        # CSS for freeze: col-letter header at top:0, data row 1 at top:28px
        self.freeze_css = widgets.HTML(value="""
<style>
.mini-excel-freeze-header {
    position: sticky !important;
    top: 0;
    z-index: 101;
    background: #f0f0f0 !important;
}
.mini-excel-freeze-header .widget-label {
    background: #f0f0f0;
}
.mini-excel-freeze-row1 {
    position: sticky !important;
    top: 28px;
    z-index: 100;
    background: white !important;
}
</style>
""")

        self.status  = widgets.HTML(value='<span style="color:#888">Ready. All columns locked — use Column Edit to enable editing.</span>')

        self.grid_box = widgets.VBox([])

        self.scroll_wrap = widgets.Box(
            [self.grid_box],
            layout=widgets.Layout(width='100%', overflow_x='auto', display='block'),
        )

        self._render_grid()

        self.container = widgets.VBox([
            self.freeze_css,
            toolbar0, toolbar1, toolbar2, toolbar3,
            toolbar4, toolbar5, toolbar6, toolbar7, toolbar8, toolbar9,
            self.scroll_wrap, self.status,
        ])

    # ── Pagination helpers ─────────────────────────────────────────────────
    def _total_pages(self):
        return max(1, -(-self.sheet.rows // self.page_size))

    def _update_page_indicator(self):
        total = self._total_pages()
        self.page_label.value = f'Page {self.page + 1} of {total}'
        self.prev_btn.disabled = (self.page == 0)
        self.next_btn.disabled = (self.page >= total - 1)

    def _on_prev_page(self, _):
        if self.page > 0:
            self.page -= 1
            self._render_grid(); self._hard_refresh()

    def _on_next_page(self, _):
        if self.page < self._total_pages() - 1:
            self.page += 1
            self._render_grid(); self._hard_refresh()

    def _on_page_size_change(self, change):
        self.page_size = change['new']
        self.page = 0
        self._render_grid(); self._hard_refresh()

    # ── Freeze top row helpers ─────────────────────────────────────────────
    def _on_freeze_toggle(self, change):
        self.freeze_top = change['new']
        if self.freeze_top:
            self.scroll_wrap.layout.max_height = f'{self.grid_height}px'
            self.scroll_wrap.layout.overflow_y = 'auto'
            self.freeze_config_row.layout.display = ''
            self.freeze_btn.button_style = 'info'
            if hasattr(self, '_header_row_widget'):
                self._header_row_widget.add_class('mini-excel-freeze-header')
            if hasattr(self, '_first_data_row_widget') and self._first_data_row_widget:
                self._first_data_row_widget.add_class('mini-excel-freeze-row1')
        else:
            self.scroll_wrap.layout.max_height = ''
            self.scroll_wrap.layout.overflow_y = ''
            self.freeze_config_row.layout.display = 'none'
            self.freeze_btn.button_style = ''
            if hasattr(self, '_header_row_widget'):
                self._header_row_widget.remove_class('mini-excel-freeze-header')
            if hasattr(self, '_first_data_row_widget') and self._first_data_row_widget:
                self._first_data_row_widget.remove_class('mini-excel-freeze-row1')

    def _on_apply_grid_height(self, _):
        self.grid_height = self.freeze_height_input.value
        if self.freeze_top:
            self.scroll_wrap.layout.max_height = f'{self.grid_height}px'
        self._set_status(f'Grid height set to {self.grid_height}px.')

    # ── Column edit whitelist helpers ──────────────────────────────────────
    def _on_toggle_col_edit_config(self, _):
        current = self.ce_config_row.layout.display
        self.ce_config_row.layout.display = 'none' if current != 'none' else ''

    def _on_enable_col_edit(self, _):
        col_str = self.ce_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter (e.g. A).'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c >= self.sheet.cols:
            self._set_status(f'Column {col_str} is out of range.'); return
        self.editable_cols.add(c)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Column {col_str} is now editable.')

    def _on_disable_col_edit(self, _):
        col_str = self.ce_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter (e.g. A).'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        self.editable_cols.discard(c)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Column {col_str} is now read-only.')

    def _on_enable_all_cols(self, _):
        self.editable_cols = set(range(self.sheet.cols))
        self._render_grid(); self._hard_refresh()
        self._set_status('All columns are now editable.')

    def _on_disable_all_cols(self, _):
        self.editable_cols.clear()
        self._render_grid(); self._hard_refresh()
        self._set_status('All columns are now read-only.')

    # ── Grid ───────────────────────────────────────────────────────────────
    def _render_grid(self):
        row_start = self.page * self.page_size
        row_end   = min(row_start + self.page_size, self.sheet.rows)

        row_layout = widgets.Layout(flex_flow='row nowrap')
        fixed40 = widgets.Layout(width='40px', min_width='40px', flex='0 0 auto',
                                 border='1px solid #ccc')
        corner = widgets.Label(value='', layout=fixed40)
        headers = [corner]
        self.col_header_widgets.clear()
        for c in range(self.sheet.cols):
            cw = self._col_w(c)
            h = widgets.Label(value=col_letter(c),
                              layout=widgets.Layout(width=cw, min_width=cw,
                                                    flex='0 0 auto',
                                                    border='1px solid #ccc'))
            self.col_header_widgets[c] = h
            headers.append(h)
        header_row = widgets.HBox(headers, layout=row_layout)
        self._header_row_widget = header_row
        if self.freeze_top:
            header_row.add_class('mini-excel-freeze-header')

        rows = [header_row]
        self._first_data_row_widget = None
        self.cell_widgets.clear()
        for i, r in enumerate(range(row_start, row_end)):
            row_widgets = [widgets.Label(value=str(r + 1), layout=fixed40)]
            for c in range(self.sheet.cols):
                current_val = self.sheet.get_display(r, c)
                layout = self._cell_layout(c)
                if c in self.editable_cols:
                    if c in self.col_dropdowns:
                        options = self.col_dropdowns[c]
                        dd_options = [''] + options
                        if current_val and current_val not in dd_options:
                            dd_options = ['', current_val] + options
                        chosen = current_val if current_val in dd_options else ''
                        w = widgets.Dropdown(options=dd_options, value=chosen, layout=layout)
                    else:
                        w = widgets.Text(value=current_val, layout=layout)
                    w._coords = (r, c)
                    w.observe(self._on_cell_change, names='value')
                else:
                    # read-only: plain label, no observer
                    w = widgets.Label(value=current_val,
                                      layout=widgets.Layout(
                                          width=layout.width, min_width=layout.min_width,
                                          flex='0 0 auto', border='1px solid #e0e0e0',
                                          background_color='#fafafa'))
                    w._coords = (r, c)
                self.cell_widgets[(r, c)] = w
                row_widgets.append(w)
            data_row = widgets.HBox(row_widgets, layout=row_layout)
            if i == 0:
                self._first_data_row_widget = data_row
                if self.freeze_top:
                    data_row.add_class('mini-excel-freeze-row1')
            rows.append(data_row)
        self.grid_box.children = rows
        for c in self.hidden_cols:
            self._apply_col_visibility(c, True)
        self._update_page_indicator()

    # ── Cell / formula bar events ──────────────────────────────────────────
    def _on_cell_change(self, change):
        if self._programmatic_update: return
        w = change['owner']; r, c = w._coords
        self.selected = (r, c)
        self.cell_label.value = rc_to_a1(r, c)
        self.sheet.set_cell(r, c, change['new'])
        self._refresh_all_cells()
        self._programmatic_update = True
        self.formula_bar.value = self.sheet.get_raw(r, c)
        self._programmatic_update = False
        self._set_status(
            f'Set {rc_to_a1(r,c)} = {self.sheet.get_raw(r,c)!r} → {self.sheet.get_display(r,c)}')

    def _on_formula_bar_change(self, change):
        if self._programmatic_update: return
        r, c = self.selected
        if c not in self.editable_cols:
            return  # formula bar blocked for locked columns
        self.sheet.set_cell(r, c, change['new'])
        self._refresh_all_cells()

    def _refresh_all_cells(self):
        self._programmatic_update = True
        for (r, c), w in self.cell_widgets.items():
            nd = self.sheet.get_display(r, c)
            if isinstance(w, widgets.Label):
                if w.value != nd:
                    w.value = nd
            elif c in self.col_dropdowns:
                if nd not in w.options:
                    new_opts = list(w.options)
                    new_opts.insert(1, nd)
                    w.options = new_opts
                if w.value != nd:
                    w.value = nd
            elif w.value != nd and (r, c) != self.selected:
                w.value = nd
        self._programmatic_update = False

    # ── Dropdown config events ─────────────────────────────────────────────
    def _on_toggle_dd_config(self, _):
        current = self.dd_config_row.layout.display
        self.dd_config_row.layout.display = 'none' if current != 'none' else ''

    def _on_apply_dropdown(self, _):
        col_str = self.dd_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter (e.g. A).'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c >= self.sheet.cols:
            self._set_status(
                f'Column {col_str} is out of range (sheet has {self.sheet.cols} columns).')
            return
        items_str = self.dd_items_input.value.strip()
        if not items_str:
            self._set_status('Enter at least one item.'); return
        options = [item.strip() for item in items_str.split(',') if item.strip()]
        if not options:
            self._set_status('No valid items found.'); return
        self.col_dropdowns[c] = options
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Dropdown applied to column {col_str}: {options}')

    def _on_remove_dropdown(self, _):
        col_str = self.dd_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter to remove its dropdown.'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c in self.col_dropdowns:
            del self.col_dropdowns[c]
            self._render_grid(); self._hard_refresh()
            self._set_status(f'Dropdown removed from column {col_str}.')
        else:
            self._set_status(f'Column {col_str} has no dropdown configured.')

    # ── Column width events ────────────────────────────────────────────────
    def _on_toggle_cw_config(self, _):
        current = self.cw_config_row.layout.display
        self.cw_config_row.layout.display = 'none' if current != 'none' else ''

    def _on_apply_col_width(self, _):
        col_str = self.cw_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter (e.g. A).'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c >= self.sheet.cols:
            self._set_status(f'Column {col_str} is out of range.'); return
        px = self.cw_width_input.value
        self.col_widths[c] = px
        self._apply_col_width(c, px)
        self._set_status(f'Column {col_str} width set to {px}px.')

    def _on_reset_col_width(self, _):
        col_str = self.cw_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter to reset.'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c in self.col_widths:
            del self.col_widths[c]
            self._apply_col_width(c, 90)
            self._set_status(f'Column {col_str} width reset to default (90px).')
        else:
            self._set_status(f'Column {col_str} is already at default width.')

    # ── Hide / Unhide column events ────────────────────────────────────────
    def _on_toggle_hide_config(self, _):
        current = self.hc_config_row.layout.display
        self.hc_config_row.layout.display = 'none' if current != 'none' else ''

    def _on_hide_col(self, _):
        col_str = self.hc_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter (e.g. A).'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c >= self.sheet.cols:
            self._set_status(f'Column {col_str} is out of range.'); return
        if c in self.hidden_cols:
            self._set_status(f'Column {col_str} is already hidden.'); return
        self.hidden_cols.add(c)
        self._apply_col_visibility(c, True)
        self._set_status(f'Column {col_str} hidden. Use Unhide to restore it.')

    def _on_unhide_col(self, _):
        col_str = self.hc_col_input.value.strip().upper()
        if not col_str:
            self._set_status('Enter a column letter (e.g. A).'); return
        try:
            c = col_index(col_str)
        except Exception:
            self._set_status(f'Invalid column: {col_str!r}'); return
        if c not in self.hidden_cols:
            self._set_status(f'Column {col_str} is not hidden.'); return
        self.hidden_cols.discard(c)
        self._apply_col_visibility(c, False)
        self._set_status(f'Column {col_str} restored.')

    # ── Toolbar events ─────────────────────────────────────────────────────
    def _on_load_data(self, _):
        text = self.input_area.value.strip()
        if not text:
            self._set_status('Nothing to load — paste data into the Input data box first.')
            return
        delimiter = '\t' if '\t' in text else ','
        reader = csv.reader(io.StringIO(text), delimiter=delimiter)
        rows = [row for row in reader if any(cell.strip() for cell in row)]
        if not rows:
            self._set_status('Could not parse data.'); return
        self.sheet.load_from_2d(rows)
        self.page = 0
        self.editable_cols.clear()
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Loaded {len(rows)} row(s), {max(len(r) for r in rows)} col(s). '
                         f'Use Column Edit to enable editing.')

    def _on_undo(self, _):
        if self.sheet.undo():
            self._hard_refresh(); self._set_status('Undid last change.')
        else:
            self._set_status('Nothing to undo.')

    def _on_redo(self, _):
        if self.sheet.redo():
            self._hard_refresh(); self._set_status('Redid change.')
        else:
            self._set_status('Nothing to redo.')

    def _on_ins_row(self, _):
        r, _ = self.selected
        self.sheet.insert_row(r)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Inserted row at {r + 1}.')

    def _on_del_row(self, _):
        r, _ = self.selected
        self.sheet.delete_row(r)
        total = self._total_pages()
        if self.page >= total:
            self.page = max(0, total - 1)
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Deleted row {r + 1}.')

    def _on_ins_col(self, _):
        _, c = self.selected
        self.sheet.insert_col(c)
        # shift editable_cols indices at or after insertion point
        self.editable_cols = {(x + 1 if x >= c else x) for x in self.editable_cols}
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Inserted column at {col_letter(c)}.')

    def _on_del_col(self, _):
        _, c = self.selected
        self.sheet.delete_col(c)
        # remove deleted col and shift remaining indices down
        self.editable_cols = {(x - 1 if x > c else x) for x in self.editable_cols if x != c}
        self._render_grid(); self._hard_refresh()
        self._set_status(f'Deleted column {col_letter(c)}.')

    def _on_save_csv(self, _):
        path = (self.path_input.value or 'miniexcel_output') + '.csv'
        self.sheet.save_csv(path); self._set_status(f'Saved → {path}')

    def _on_save_xlsx(self, _):
        path = (self.path_input.value or 'miniexcel_output') + '.xlsx'
        self.sheet.save_xlsx(path); self._set_status(f'Saved → {path}')

    def _on_clear(self, _):
        self.sheet = Spreadsheet(self.sheet.rows, self.sheet.cols)
        self.page = 0
        self.editable_cols.clear()
        self._render_grid(); self._hard_refresh(); self._set_status('Cleared.')

    def _hard_refresh(self):
        self._programmatic_update = True
        for (r, c), w in self.cell_widgets.items():
            nd = self.sheet.get_display(r, c)
            if isinstance(w, widgets.Label):
                if w.value != nd:
                    w.value = nd
            elif c in self.col_dropdowns:
                options = list(w.options)
                if nd not in options:
                    options.insert(1, nd)
                    w.options = options
                w.value = nd if nd in w.options else ''
            else:
                w.value = nd
        self._programmatic_update = False
        self._update_page_indicator()

    def _set_status(self, msg):
        self.status.value = f'<span style="color:#333">{msg}</span>'

    def load_sample(self):
        sample = [
            ['Product', 'Q1',   'Q2',   'Q3',   'Q4',   'Total',          'Avg'],
            ['Widgets', '1200', '1450', '1600', '1800', '=SUM(B2:E2)',   '=AVERAGE(B2:E2)'],
            ['Gadgets', '800',  '950',  '1100', '1300', '=SUM(B3:E3)',   '=AVERAGE(B3:E3)'],
            ['Gizmos',  '500',  '600',  '750',  '900',  '=SUM(B4:E4)',   '=AVERAGE(B4:E4)'],
            ['Doodads', '300',  '350',  '400',  '450',  '=SUM(B5:E5)',   '=AVERAGE(B5:E5)'],
            ['Total', '=SUM(B2:B5)', '=SUM(C2:C5)', '=SUM(D2:D5)', '=SUM(E2:E5)',
             '=SUM(F2:F5)', '=AVERAGE(G2:G5)'],
            [],
            ['Stats', 'Value'],
            ['Best quarter Q4 total', '=E6'],
            ['Growth Q1->Q4 (Widgets)', '=(E2-B2)/B2'],
            ['High performer?', '=IF(F2>5000, "Yes", "No")'],
        ]
        self.sheet.load_from_2d(sample)
        self.page = 0
        self.editable_cols.clear()
        self._render_grid(); self._hard_refresh()
        self._set_status('Sample data loaded. Use Column Edit → Enable to make columns editable.')

    def show(self):
        display(self.container)

app = MiniExcelUI(rows=14, cols=8)
app.load_sample()
app.show()

## How to use

- **Edit a cell**: click a cell and type. Press Tab or Enter to commit.
- **Formulas**: start with `=`. Examples: `=A1+B1`, `=SUM(A1:A5)`, `=IF(A1>10, "big", "small")`.
- **Structural edits**: click a cell, then use `+Row`, `-Row`, `+Col`, `-Col`.
- **Undo / redo**: the toolbar buttons.
- **Save**: pick a filename, then click Save CSV or Save XLSX. Files land in your notebook's working directory.
- **Clear All**: wipes the sheet (undo-able).

### Pagination

The grid only renders the current page of rows — this keeps the UI fast even with large datasets.

- **Rows/page**: choose 10, 20, 30, 40, or 50 rows per page from the dropdown.
- **◀ Prev / Next ▶**: navigate between pages.
- **Page indicator**: shows current page and total (e.g. `Page 3 of 20`). Prev/Next auto-disable at the boundaries.
- Row numbers always reflect their **absolute position** in the sheet (e.g. row 51 on page 2 shows `51`, not `1`).
- Cell references in formulas (`A51`, `=SUM(A1:A100)`) span the whole sheet regardless of which page is visible.

### Freeze Top Row

Keeps the column-letter header (A, B, C…) visible while scrolling down through a tall page.

1. Click **Freeze Top Row** — the button turns blue and the grid area gets a fixed height with a vertical scrollbar.
2. Scroll down inside the grid: the header row stays pinned at the top.
3. A **Height px** spinner appears next to the button (default 400 px, range 100–800). Change the value and click **Apply** to resize the scrollable area.
4. Click **Freeze Top Row** again to toggle it off and restore the full-height grid.

> **Tip:** Combine with pagination — set a larger rows/page (e.g. 50) and enable Freeze Top Row so the header is always visible as you scroll through the page.

### Column Dropdowns

1. Click **Add Dropdown** to expand the dropdown configurator.
2. Type a column letter in **Column** (e.g. `A`, `B`).
3. Type a comma-separated list of choices in **Items** (e.g. `Yes, No, Maybe`).
4. Click **Apply** — every cell in that column becomes a dropdown select.
5. To revert to free-text, enter the column letter and click **Remove**.

Existing cell values not in the list are preserved as an extra option.

### Column Width

1. Click **Column Width** to expand the width configurator.
2. Type a column letter in **Column** (e.g. `A`).
3. Set the desired width in pixels using the **Width px** spinner (30–600 px, default 90).
4. Click **Apply** — the header and every cell in that column resize immediately.
5. Click **Reset** to restore the column to the 90 px default.

### Hide / Unhide Column

1. Click **Hide/Unhide Col** to expand the configurator.
2. Type a column letter in **Column** (e.g. `B`).
3. Click **Hide** — the column header and all its cells disappear from view. The data is preserved.
4. To restore, type the same column letter and click **Unhide**.

Hidden columns are not affected by insert/delete row operations and survive `_render_grid` rebuilds. Note: hidden-column state is not persisted to CSV/XLSX.

## Supported functions
`SUM`, `AVERAGE`, `MIN`, `MAX`, `COUNT`, `PRODUCT`, `ABS`, `ROUND`, `IF`, `CONCAT` / `CONCATENATE`

Operators: `+ - * / ^`, comparisons `= <> < > <= >=`, parentheses, unary `-`.

## Known limitations
- Cell formulas do NOT auto-adjust when you insert/delete rows or columns (references are not rewritten).
- No multi-cell selection, copy/paste yet.
- Dropdown configurations, column widths, hidden-column state, and freeze state are not persisted to CSV/XLSX.